# General Purpose English Next Word Predictor using LSTM

This notebook trains a simple next-word predictor using PyTorch.

It uses normal English sentences, not Shakespeare or random generated text.

Given a phrase like:

`machine learning is`

the model tries to predict the next word.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import re
import requests


## 1. Download a simple English dataset

We use a small public text dataset containing normal English stories from Project Gutenberg.

In [ ]:
url = "https://www.gutenberg.org/files/11/11-0.txt"  # Alice in Wonderland

response = requests.get(url)
text = response.text

print("Downloaded characters:", len(text))
print(text[:500])


## 2. Clean the text

We convert everything to lowercase and keep only English words.

In [ ]:
text = text.lower()

# keep only letters, spaces, and basic punctuation
text = re.sub(r"[^a-z\.\?\!\s]", " ", text)
text = re.sub(r"\s+", " ", text)

words = text.split()

print("Total words:", len(words))
print(words[:50])


## 3. Limit vocabulary

To make training faster and easier, we keep only the most common words.

In [ ]:
from collections import Counter

vocab_size = 3000

word_counts = Counter(words)
most_common = word_counts.most_common(vocab_size - 1)

vocab = ["<UNK>"] + [word for word, count in most_common]

word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for word, i in word_to_idx.items()}

encoded_words = [word_to_idx.get(word, word_to_idx["<UNK>"]) for word in words]

print("Vocabulary size:", len(vocab))
print(vocab[:30])


## 4. Create training examples

The model sees 5 words and predicts the 6th word.

In [ ]:
seq_len = 5

X = []
y = []

for i in range(len(encoded_words) - seq_len):
    X.append(encoded_words[i:i + seq_len])
    y.append(encoded_words[i + seq_len])

X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

print("Input shape:", X.shape)
print("Target shape:", y.shape)

print("Example input:", [idx_to_word[i.item()] for i in X[0]])
print("Target word:", idx_to_word[y[0].item()])


## 5. Define the LSTM model

In [ ]:
class NextWordLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=128):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        output, _ = self.lstm(x)

        last_output = output[:, -1, :]
        prediction = self.fc(last_output)

        return prediction


## 6. Train the model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = NextWordLSTM(len(vocab)).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Using device:", device)


In [ ]:
batch_size = 128
epochs = 5

for epoch in range(epochs):
    total_loss = 0

    permutation = torch.randperm(X.size(0))

    for i in range(0, X.size(0), batch_size):
        indices = permutation[i:i + batch_size]

        batch_x = X[indices].to(device)
        batch_y = y[indices].to(device)

        outputs = model(batch_x)
        loss = loss_fn(outputs, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1}, Loss: {total_loss:.4f}")


## 7. Predict the next word

In [ ]:
def clean_input(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r"[^a-z\s]", " ", sentence)
    sentence = re.sub(r"\s+", " ", sentence)
    return sentence.strip().split()


def predict_next_word(sentence, top_k=5):
    model.eval()

    input_words = clean_input(sentence)

    if len(input_words) < seq_len:
        return "Please enter at least 5 words."

    input_words = input_words[-seq_len:]
    input_ids = [word_to_idx.get(word, word_to_idx["<UNK>"]) for word in input_words]

    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        probabilities = torch.softmax(output, dim=1)
        top_values, top_indices = torch.topk(probabilities, top_k)

    predictions = []
    for idx, prob in zip(top_indices[0], top_values[0]):
        predictions.append((idx_to_word[idx.item()], round(prob.item(), 4)))

    return predictions


## 8. Try examples

In [ ]:
examples = [
    "alice was beginning to get very",
    "the rabbit hole went straight on",
    "she opened the door and found",
    "i do not know what to",
    "it was a very curious"
]

for sentence in examples:
    print("Input:", sentence)
    print("Predictions:", predict_next_word(sentence))
    print()


## 9. Type your own sentence

In [ ]:
sentence = input("Enter at least 5 words: ")
print("Top predictions:", predict_next_word(sentence))


## 10. Generate multiple words

This repeatedly predicts the next word and appends it to the sentence.

In [ ]:
def generate_words(start_sentence, number_of_words=20):
    sentence = start_sentence

    for _ in range(number_of_words):
        predictions = predict_next_word(sentence, top_k=1)

        if isinstance(predictions, str):
            return predictions

        next_word = predictions[0][0]
        sentence += " " + next_word

    return sentence


print(generate_words("alice was beginning to get very", 20))


## Explanation

This is a next-word prediction model. It reads 5 previous words and predicts the most likely next word.

The model has three main parts:

1. **Embedding layer**: converts each word into a numerical vector.
2. **LSTM layer**: learns the order and context of words in a sentence.
3. **Linear layer**: chooses the next word from the vocabulary.

This is a simple version of how predictive typing works.